# 🎭 Nuance Talking Avatar — Real-Time Cloud GPU Runner (LivePortrait)

This notebook runs the **Nuance-style Talking Avatar** system with **NVIDIA T4 GPU acceleration** for free on Google Colab.

### Capabilities:
- **High-Speed Real-Time Synthesis**: ~30–35 FPS inference (~25ms/frame) powered by NVIDIA T4 CUDA acceleration.
- **Active Listening & Micro-Expressions**: Real-time controls for head pitch/yaw/roll micro-nods, eye retargeting, eyeball saccades, smile, and lip articulation.
- **Native Inline UI**: Renders the complete interactive UI directly inside this notebook window.

> **Step 1:** Make sure your Colab runtime is set to **T4 GPU**:
> Click **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify GPU Hardware Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    raise SystemError("GPU not detected! Please go to Runtime > Change runtime type > select T4 GPU.")

In [ ]:
# 2. Clone Your Nuance Avatar Repository
import os
%cd /content
if os.path.exists("/content/LivePortrait"):
    !rm -rf /content/LivePortrait
!git clone https://github.com/samuelQUANSAH/nuance-talking-avatar.git /content/LivePortrait
%cd /content/LivePortrait

In [ ]:
# 3. Install GPU Dependencies
!pip install -q onnx onnxruntime-gpu opencv-python-headless tyro gradio pyyaml imageio imageio-ffmpeg scipy pykalman

In [ ]:
# 4. Download Pretrained Neural Weights (~2.5GB in ~20s on Colab)
import os
from huggingface_hub import snapshot_download

os.makedirs("pretrained_weights", exist_ok=True)
print("Streaming foundation weights from KlingTeam/LivePortrait...")
snapshot_download(
    repo_id="KlingTeam/LivePortrait",
    local_dir="pretrained_weights",
    ignore_patterns=["*.git*", "README.md", "docs/*", "*animals*"],
    local_dir_use_symlinks=False
)
print("Weights successfully initialized!")

In [ ]:
# 5. Verify Avatar Assets & Pretrained Checkpoints
import os
avatar_path = "assets/examples/source/avatar_source.jpg"
if os.path.exists(avatar_path):
    print(f"[✓] Avatar source ready: {avatar_path} ({os.path.getsize(avatar_path)} bytes)")
else:
    print("[!] Avatar source not found in repo, fetching from cloud backup...")
    import urllib.request
    os.makedirs("assets/examples/source", exist_ok=True)
    urllib.request.urlretrieve(
        "https://huggingface.co/spaces/samuel-QUANSAH/citronCALAPP/resolve/main/assets/examples/source/avatar_source.jpg",
        avatar_path
    )
    print(f"[✓] Restored avatar source: {avatar_path}")

print("System ready for real-time inference!")

In [ ]:
# 6. Launch Native Inline Avatar Web Interface (Option A)
import sys
import os
%cd /content/LivePortrait

# Import the prepared Gradio demo directly into notebook memory
from app import demo

# Launch inline directly inside Google Colab (no external subprocess, no tunnel hang)
demo.launch(
    inline=True,
    share=True,
    server_port=7860,
    server_name="0.0.0.0",
    show_error=True
)